# Week 1-2

In [ ]:
import math

# ── CHANGE ONLY THIS ──────────────────────
RIM_RADIUS   = 0.3   # metres  ← your one parameter
# ─────────────────────────────────────────

N            = 16
CENTER       = (6.1, 0, 3)
TUBE_RADIUS  = 0.02
RGBA         = "1.0 0.4 0.0 1"
CONAFFINITY  = "1"
CONTYPE      = "0"

seg_half = math.pi * RIM_RADIUS / N
cx, cy, cz = CENTER

print(f"<!-- Rim: R={RIM_RADIUS}, N={N}, tube={TUBE_RADIUS} -->")
for i in range(N):
    theta = 2 * math.pi * i / N
    px = cx + RIM_RADIUS * math.cos(theta)
    py = cy + RIM_RADIUS * math.sin(theta)
    ax = -math.cos(theta)
    ay = -math.sin(theta)
    print(
        f'<geom type="cylinder" '
        f'size="{TUBE_RADIUS} {seg_half:.5f}" '
        f'pos="{px:.4f} {py:.4f} {cz}" '
        f'axisangle="{ax:.4f} {ay:.4f} 0 1.5708" '
        f'conaffinity="{CONAFFINITY}" contype="{CONTYPE}" '
        f'rgba="{RGBA}"/>'
    )

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Reacher-v5", xml_file =r"C:\vsr\Naren\FAFO-RL\agnet_xml\reacher.xml", render_mode = "human")
observation, info = env.reset(seed=42)


print("Obs shape:", observation.shape)
print("Action space:", env.action_space)

try:
    for i in range(1000):  # just 5 steps to see info
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i} | reward: {reward:.4f} | info: {info}")

        if terminated or truncated:
            observation, info = env.reset()
finally:
    env.close()

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Pendulum-v1")
model = SAC("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=20000)
print("Stack working")

In [ ]:
env = gym.make("Pendulum-v1", render_mode = "human")


obs, info = env.reset(seed=42)

try:
    total_reward = 0
    for i in range(1000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
            print(total_reward)
            observation, info = env.reset()
            total_reward = 0
finally:
    env.close()

## Reacher SAC

In [ ]:
import numpy as np
import gymnasium as gym
class CustomPendulumEnv(gym.Wrapper):
    def angle_normalize(self, x):
        return ((x + np.pi) % (2 * np.pi)) - np.pi

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # obs = [cos(th), sin(th), thdot]
        th = np.arctan2(obs[1], obs[0])   # recover angle from cos/sin
        thdot = obs[2]
        u = np.clip(action, -2.0, 2.0)[0]

        # your custom reward
        costs = self.angle_normalize(th)**2 + 0.1 * thdot**2
        modified_reward = -costs           # dropped action cost term

        return obs, modified_reward, terminated, truncated, info

In [ ]:
import inspect
# import gymnasium as gym

# Create the base environment (unwrapped to bypass time limits/wrappers)
env = CustomPendulumEnv(gym.make("Pendulum-v1"))

# Print the source code of the step function where the reward is calculated
print(inspect.getsource(env.step))

In [ ]:
from stable_baselines3 import SAC, PPO
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=20e3, tb_log_name="pendulum_modifield_reward")
print("Stack working")

## Different Seed Testing

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO
env = gym.make("Reacher-v5")
# get_device('cpu')
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=500e3, tb_log_name="reacher_sac_long")
# model = SAC("MlpPolicy", env, verbose=1,seed = 33, device="cpu", tensorboard_log = r".\log")
# model.learn(total_timesteps=20e3, tb_log_name="pendulum_different_seed")
print("Stack working")

In [ ]:
model.save(r".\models\reacher_sac_500k")

In [ ]:
del model

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import SAC

# class MountainCarWrapper(gym.Wrapper):
#     def step(self, action):
#         obs, reward, terminated, truncated, info = self.env.step(action)
#         position, velocity = obs
#         reward += 10 * abs(velocity)
#         reward += 3 * (position + 0.5)
#         return obs, reward, terminated, truncated, info

# env = MountainCarWrapper(gym.make("MountainCarContinuous-v0"))
env = gym.make("MountainCarContinuous-v0")
model = SAC("MlpPolicy", env,
            learning_starts=1000,
            verbose=1,
            tensorboard_log=r".\log",
            seed=42)
model.learn(total_timesteps=100_000,
            tb_log_name="SAC_MountainCar_normal")

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO

model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=100e3, tb_log_name="mountain_car_continuous_reward_wrapper")
print("Stack working")

In [ ]:
# import gymnasium as gym
# from stable_baselines3 import SAC, PPO

env = gym.make("MountainCarContinuous-v0", render_mode = "human")
# model = SAC.load(r".\models\reacher_sac_500k", env=env)

# obs, info = env.reset(seed=32)
obs, info = env.reset()

try:
    total_reward = 0
    for i in range(10000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
        # print(total_reward)
            observation, info = env.reset()
        total_reward = 0
finally:
    env.close()

# Week 3

In [ ]:
import mujoco

model = mujoco.MjModel.from_xml_path(
    r"E:\personal projects\FAFO-RL\agnet_xml\three_pointer.xml"
)
data = mujoco.MjData(model)
print("Bodies  :", model.nbody)
print("Joints  :", model.njnt)
print("Actuators:", model.nu)
print("qpos shape:", data.qpos.shape)
print("Valid ✓")

In [ ]:
# import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC
from pathlib import Path
from stable_baselines3.common.env_checker import check_env
# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)
# current_dir = Path.cwd()
# xml_path_str = str(current_dir.parent / "agnet_xml" / "three_pointer.xml")
env = gym.make(
    "ThreePointer-v0",
    # xml_file=xml_path_str,
    render_mode="human",
    
)
check_env(env)
# observation, info = env.reset(seed=42)
# print("Obs shape:", observation.shape)
# print("Action space:", env.action_space)

# try:
#     for i in range(10000):
#         action = env.action_space.sample()
#         observation, reward, terminated, truncated, info = env.step(action)
#         print(f"{i}", f"Observation {observation[7]}")

#         if terminated or truncated:
#             observation, info = env.reset()
# finally:
#     env.close()

In [1]:
# import mujoco
from stable_baselines3 import SAC, PPO
from stable_baselines3.common.env_util import make_vec_env
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC
from pathlib import Path

# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).

register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

env = gym.make(
    "ThreePointer-v0",
    # render_mode="human"
)
model = SAC("MlpPolicy", env, verbose=1, device="cuda", tensorboard_log = r".\three_pointer_log")
model.learn(total_timesteps=1000e3, tb_log_name="default_settings")
print("Stack working")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to .\three_pointer_log\default_settings_4
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 24.8      |
|    ep_rew_mean     | -1.03e+04 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 6188      |
|    time_elapsed    | 0         |
|    total_timesteps | 99        |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 25.8      |
|    ep_rew_mean     | -1.03e+04 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 408       |
|    time_elapsed    | 0         |
|    total_timesteps | 206       |
| train/             |           |
|    actor_loss      | 94.8      |
|    critic_loss     | 2.7e+06   |
|    ent_coef        | 0.969     |
|    ent_coef_loss   | -0.0979   |
|    learning_ra

KeyboardInterrupt: 

In [2]:
model.save(r".\models\three_pointer_sac_1000k")


In [ ]:
import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC

# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

env = gym.make(
    "ThreePointer-v0",
    render_mode="human",
)
model = SAC.load(r"C:\vsr\Naren\FAFO-RL\scripts\models\three_pointer_sac_1000k.zip", env=env)
observation, info = env.reset()          # unpack the tuple — this was the crash
try:
    for i in range(10000):
        action, _states = model.predict(observation, deterministic=True)
        observation, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            print(f'{i}' ,f"terminated{terminated}", f"truncated {truncated}")

            observation, info = env.reset()
finally:
    env.close()

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
0 terminatedFalse truncated False
1 terminatedFalse truncated False
2 terminatedFalse truncated False
3 terminatedFalse truncated False
4 terminatedFalse truncated False
5 terminatedFalse truncated False
6 terminatedFalse truncated False
7 terminatedFalse truncated False
8 terminatedFalse truncated False
9 terminatedFalse truncated False
10 terminatedFalse truncated False
11 terminatedFalse truncated False
12 terminatedFalse truncated False
13 terminatedFalse truncated False
14 terminatedFalse truncated False
15 terminatedFalse truncated False
16 terminatedFalse truncated False
17 terminatedFalse truncated False
18 terminatedFalse truncated False
19 terminatedFalse truncated False
20 terminatedFalse truncated False
21 terminatedFalse truncated False
22 terminatedFalse truncated False
23 terminatedFalse truncated False
24 terminatedFalse truncated False
25 terminatedFalse truncated False
26 terminatedFalse trun

In [1]:
import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC

# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

env = gym.make(
    "ThreePointer-v0",
    render_mode="human",
)
observation, info = env.reset()   

try:
    for i in range(1000):  # just 5 steps to see info
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i} | terminated: {terminated:.4f} | observation: {observation[4]} | Reward {reward}")

        if terminated or truncated:
            observation, info = env.reset()
finally:
    env.close()

Step 0 | terminated: 0.0000 | observation: 1.302450360765141 | Reward -11.44324539340392
Step 1 | terminated: 0.0000 | observation: 1.306540313020046 | Reward -11.670130248759225
Step 2 | terminated: 0.0000 | observation: 1.3073968510101384 | Reward -11.613284342687711
Step 3 | terminated: 0.0000 | observation: 1.3043293890002308 | Reward -11.356856433115768
Step 4 | terminated: 0.0000 | observation: 1.2973379269903231 | Reward -11.77314580960389
Step 5 | terminated: 0.0000 | observation: 1.2864224649804155 | Reward -11.875498089054862
Step 6 | terminated: 0.0000 | observation: 1.271583002970508 | Reward -11.379617063441362
Step 7 | terminated: 0.0000 | observation: 1.2528195409606004 | Reward -11.6480124227531
Step 8 | terminated: 0.0000 | observation: 1.2301320789506929 | Reward -11.331054750577335
Step 9 | terminated: 0.0000 | observation: 1.2035206169407853 | Reward -11.987578855967596
Step 10 | terminated: 0.0000 | observation: 1.1729851549308776 | Reward -11.354223580608402
Step 

KeyboardInterrupt: 